# Stabilis: fine-tuning do extrator no Kaggle (trilha LLM, etapa L3)

Treina o modelo que lê textos de pacientes e anamneses (QLoRA com Unsloth), gera o GGUF Q4_K_M e envia para um
repositório **privado** no Hugging Face, mostrando as três variáveis da promoção.

**Antes de rodar**
- Acelerador **GPU T4 ×2** e internet ligada (Settings do notebook).
- Segredo `HF_TOKEN` (Add-ons → Secrets) com permissão de escrita no repositório privado do modelo.
- Repositório do GitHub privado: anexe também o dataset privado com `train_qlora.py` (`CODE_DIR`); o notebook copia
  o script de lá em vez de clonar.
- Dataset privado do Kaggle com `train.jsonl` e `validation.jsonl` gerados por `finetune/prepare.py` (dados sintéticos;
  **nunca** dados reais de pacientes nem o gold set).
- Confira a licença do modelo base no model card (Apache 2.0 de preferência) e se a versão fixada do
  llama-cpp-python do Space suporta a arquitetura.

Nada daqui é promovido automaticamente: depois do treino, siga "Promoção, passo a passo" em `eval/README.md`.

In [ ]:
# Configuração: ajuste antes de rodar
BASE_MODEL = "unsloth/gemma-4-E2B-it"               # escolhido na etapa 4 (eval/reports/README.md)
CHAT_TEMPLATE = "gemma-4"                           # template do Unsloth para os Gemma 4 pequenos
DATA_DIR = "/kaggle/input/stabilis-extracao-v1"     # dataset privado com train.jsonl e validation.jsonl
HF_MODEL_REPO = "amaroguilherme/stabilis-extrator"  # repositório PRIVADO de destino
CODE_DIR = "/kaggle/input/stabilis-codigo"          # dataset privado com train_qlora.py (o repositório é privado)
REPO_URL = "https://github.com/amaroguilherme/stabilis.git"  # só se CODE_DIR não estiver anexado (repositório público)
OUT_DIR = "/kaggle/working/run"
SEED = 3407

In [ ]:
!pip install -q unsloth trl datasets huggingface_hub
import os, shutil, subprocess

# O repositório do GitHub é privado: o script vem do dataset CODE_DIR. Com um repositório público, clona.
if os.path.isdir(CODE_DIR):
    os.makedirs("stabilis/finetune", exist_ok=True)
    shutil.copy(os.path.join(CODE_DIR, "train_qlora.py"), "stabilis/finetune/train_qlora.py")
else:
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, "stabilis"], check=True)

In [ ]:
import os
from kaggle_secrets import UserSecretsClient

os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
for split in ("train", "validation"):
    with open(f"{DATA_DIR}/{split}.jsonl", encoding="utf-8") as f:
        print(split, sum(1 for line in f if line.strip()), "exemplos")

In [ ]:
!cd stabilis && python finetune/train_qlora.py --base {BASE_MODEL} --chat-template {CHAT_TEMPLATE} --data {DATA_DIR} --out {OUT_DIR} --seed {SEED}

In [ ]:
# Envia o GGUF e o run.json num único commit do repositório privado e mostra as variáveis da promoção.
import glob
from huggingface_hub import CommitOperationAdd, HfApi, create_repo

token = os.environ["HF_TOKEN"]
ggufs = sorted(glob.glob(f"{OUT_DIR}/**/*.gguf", recursive=True))
assert ggufs, "nenhum GGUF encontrado: confira a saída do treino"
gguf = next((g for g in ggufs if "q4_k_m" in g.lower()), ggufs[0])
create_repo(HF_MODEL_REPO, private=True, exist_ok=True, token=token)
info = HfApi(token=token).create_commit(
    repo_id=HF_MODEL_REPO,
    operations=[
        CommitOperationAdd(path_in_repo=os.path.basename(gguf), path_or_fileobj=gguf),
        CommitOperationAdd(path_in_repo="run.json", path_or_fileobj=f"{OUT_DIR}/run.json"),
    ],
    commit_message=f"Extractor fine-tuned from {BASE_MODEL} (seed {SEED})",
)
print("MODEL_REPO_ID =", HF_MODEL_REPO)
print("MODEL_FILENAME =", os.path.basename(gguf))
print("MODEL_REVISION =", info.oid)

## Depois do treino

1. Anote as três variáveis acima.
2. Avalie com o gold set congelado (`eval/README.md`, "Promoção, passo a passo"): no seu computador,
   `run_eval.py --gates --baseline` com `EXTRACTION_PROMPT=short`; no Space, `bench_space.py --gates`.
3. Só troque as variáveis do Space se todos os portões passarem, inclusive o subconjunto crítico de risco.
4. Apague a saída deste notebook se ela tiver qualquer amostra de texto.